# 03 — FP&A Foundations: OPEX, COGS, GM, EBITDA

This notebook builds on the scenario engine and revenue model to create the next core FP&A modules:

- **OPEX Module** (scenario inflation)
- **COGS Module** (COGS% × Revenue)
- **Gross Margin**
- **EBITDA**


In [ ]:
import xarray as xr
import pandas as pd

from src.modules.scenario import ScenarioManager
from src.modules.revenue import build_revenue_module
from src.modules.opex import build_opex_module
from src.modules.cogs import build_cogs_module
from src.modules.ebitda import build_ebitda_module

## 1. Base Price and Volume (from Day 2)

In [ ]:
time = pd.period_range('2026Q1', periods=4, freq='Q')
products = ['Core', 'Premium']

price = xr.DataArray(
    [[45000, 90000],
     [45500, 91000],
     [46000, 92000],
     [46500, 93000]],
    dims=("time", "product"),
    coords={"time": time, "product": products}
)

volume = xr.DataArray(
    [[6000, 1500],
     [6050, 1525],
     [6100, 1550],
     [6150, 1575]],
    dims=("time", "product"),
    coords={"time": time, "product": products}
)

## 2. Scenarios (Base, Upside, Downside)

In [ ]:
scenarios = ['Base', 'Upside', 'Downside']

scenario_mgr = ScenarioManager(
    scenarios=scenarios,
    price_multipliers=[1.00, 1.02, 0.98],
    volume_multipliers=[1.00, 1.05, 0.95]
)

prices_s = scenario_mgr.apply_price(price)
volumes_s = scenario_mgr.apply_volume(volume)

rev_ds = build_revenue_module(prices_s, volumes_s)
rev_ds

## 3. OPEX Module
Base OPEX with scenario inflation.

In [ ]:
cost_categories = ['People', 'G&A', 'IT', 'Marketing']

base_opex = xr.DataArray(
    [[22e6, 8e6, 6e6, 10e6],
     [22e6, 8e6, 6e6, 10e6],
     [22e6, 8e6, 6e6, 10e6],
     [22e6, 8e6, 6e6, 10e6]],
    dims=("time", "cost_category"),
    coords={"time": time, "cost_category": cost_categories}
)

inflation = {'Base': 1.00, 'Upside': 1.03, 'Downside': 1.08}

opex_ds = build_opex_module(base_opex, inflation, scenarios)
opex_ds

## 4. COGS Module

In [ ]:
cogs_ds = build_cogs_module(
    revenue=rev_ds["Revenue"],
    cogs_pct=0.72
)
cogs_ds

## 5. EBITDA Module

In [ ]:
ebitda_ds = build_ebitda_module(cogs_ds, opex_ds)
ebitda_ds